In [ ]:
import torch
from tqdm import tqdm
from matplotlib import pyplot as plt
from spikingjelly.activation_based import neuron, functional, surrogate

In [ ]:
def NeuronNode():
    return neuron.LIFNode(surrogate_function=surrogate.ATan(), v_threshold=1.0, v_reset=0.0, detach_reset=False, tau=2., decay_input=False)

# def NeuronNode():
#     return neuron.LIFNode(surrogate_function=surrogate.ATan(), v_threshold=1.0, v_reset=0.0, detach_reset=False, tau=2., decay_input=True)

# def NeuronNode():
    # return neuron.ParametricLIFNode(surrogate_function=surrogate.ATan(), v_threshold=1.0, v_reset=0.0, detach_reset=True, tau=2., decay_input=True)

In [ ]:
def caldata(Timestep=16, Batch_size=256, var=0.5):

    net = NeuronNode()
    functional.set_step_mode(net, step_mode='m')
    img = torch.randn((Timestep, Batch_size, 3, 224, 224)) * (var ** 0.5)
    # print(f'img.mean() = {img.mean()}, img.var() = {img.var()}')
    with torch.no_grad():
        out = net(img)
        out_mean = out.mean(dim=[1,2,3,4])
        out_var = out.var(dim=[1,2,3,4])
        out_gamma = 1 / (out_var ** 0.5)

    return out_mean, out_var, out_gamma

In [ ]:
def mean_data(Timestep=16, Batch_size=256, var=0.5):
    out_mean, out_var, out_gamma = caldata(Timestep=Timestep, Batch_size=Batch_size, var=var)
    return out_mean.mean(), out_var.mean(), out_gamma.mean()

In [ ]:
mean, var, gamma = mean_data(Timestep=4, Batch_size=256, var=1.)
print(f'T=4: mean = {mean}, var = {var}, gamma = {gamma}')
mean, var, gamma = mean_data(Timestep=8, Batch_size=256, var=1.)
print(f'T=8: mean = {mean}, var = {var}, gamma = {gamma}')
mean, var, gamma = mean_data(Timestep=16, Batch_size=256, var=1.)
print(f'T=16: mean = {mean}, var = {var}, gamma = {gamma}')

In [ ]:
All_out_mean = []
All_out_var = []
All_out_gamma = []

var_all = torch.linspace(0, 5, 51)

In [ ]:
for i in tqdm(range(len(var_all))):
    out_mean, out_var, out_gamma = caldata(Timestep=16, Batch_size=16, var=var_all[i])
    All_out_mean.append(out_mean)
    All_out_var.append(out_var)
    All_out_gamma.append(out_gamma)
    
All_out_mean = torch.stack(All_out_mean)
All_out_var = torch.stack(All_out_var)
All_out_gamma = torch.stack(All_out_gamma)

In [ ]:
def plot_fr(data_x, data_y, ylim=None, lable_size=20, title_size=20, ticks_size=20, 
            small_ticks_size=2., linewidth=2., fig_size=(14, 5), dpi=600, grid=True, title=r'LIF ($\tau=2$) with/without Delay Input'):

    plt.figure(figsize=fig_size, dpi=dpi)
    plt.rcParams['font.sans-serif'] = 'Arial'

    for i in range(data_y.shape[1]):
        plt.plot(data_x, data_y[:, i], linewidth=linewidth, label=f'T={i+1}')
    plt.xlim((data_x.min(), data_x.max()))
    plt.legend(fontsize=10)
    plt.title(title, fontsize=title_size)
    if ylim is not None:
        plt.ylim(ylim)
    plt.xlabel('$\sigma_{in}^2$', fontsize=lable_size)
    plt.ylabel('$\sigma_{out}^2$', fontsize=lable_size)
    plt.xticks(fontsize=ticks_size)
    plt.yticks(fontsize=ticks_size)
    plt.gca().spines['top'].set_linewidth(small_ticks_size)
    plt.gca().spines['bottom'].set_linewidth(small_ticks_size)
    plt.gca().spines['left'].set_linewidth(small_ticks_size)
    plt.gca().spines['right'].set_linewidth(small_ticks_size)
    plt.gca().tick_params('both', which='major', width=small_ticks_size, direction='in')
    plt.grid(grid)

    plt.show()

In [ ]:
plot_fr(var_all, All_out_var, ylim=(0, 0.25), fig_size=(6.5, 5), title=r'LIF Node ($\tau=2$) without Decay Input')